## Introdução


Este notebook traz uma análise mais profunda dos dados disponibilizados. A intenção é ir além dos exercícios do desafio e desenvolver uma análise mais completa e que possa ser utilizada para gerar _insights_ para tomadas de decisão.

Conforme explicado durante a resolução dos exercícios, a base de dados foi criada apenas para esse projeto e com eventos fictícios. Além disso, o banco de dados já foi criado naquela etapa. Dessa forma, esta análise começa já consumindo do banco de dados.

Esta análise foca em entender como os produtos e categorias performam em vendas ao longo do ano. O objetivo é identificar tendências e padrões que permitam corrgiri falhas e explorar bons desempenhos.

## Preparação

O primeiro passo é a instalação das bibiliotecas necessárias para o correto funcionamento do projeto. Pode ser feito usando `pip install requirements.txt`

In [ ]:
# importar as bibliotecas necessárias
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

In [2]:
# carregar os dados do banco de dados para um dataframe

# abrir a conexão com o banco de dados
conn = sqlite3.connect('analise_vendas.db')

# criar a query
query = """ 
    SELECT * FROM vendas
"""

# armazenar em um dataframe
df = pd.read_sql_query(query, con=conn)

# fechar a conexão
conn.close()

# exibir o resultado
df.head()

,id_venda,data_venda,cliente,produto,categoria,quantidade,preco_unitario,vendedor,cidade,estado
0,1,2024-09-27,Pietra,Mouse,Eletrônicos,3,84.74,Ana,Belo Horizonte,RJ
1,2,2024-01-21,Vinicius,Monitor,Eletrônicos,3,950.34,Bruno,São Paulo,RS
2,3,2024-08-02,Yago,Mouse,Eletrônicos,3,79.13,Eduardo,Curitiba,SP
3,4,2024-04-09,Lara,Notebook,Eletrônicos,3,3471.47,Ana,Rio de Janeiro,RJ
4,5,2024-05-29,Levi,Fone,Eletrônicos,4,208.32,Eduardo,São Paulo,RS


In [3]:
# conferir a integridade dos dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_venda        10000 non-null  int64  
 1   data_venda      10000 non-null  str    
 2   cliente         10000 non-null  str    
 3   produto         10000 non-null  str    
 4   categoria       10000 non-null  str    
 5   quantidade      10000 non-null  int64  
 6   preco_unitario  10000 non-null  float64
 7   vendedor        10000 non-null  str    
 8   cidade          10000 non-null  str    
 9   estado          10000 non-null  str    
dtypes: float64(1), int64(2), str(7)
memory usage: 781.4 KB


In [ ]:
# tratamento da coluna de data para o formato data
df['data_venda'] = pd.to_datetime(df['data_venda'])

df.info()

In [ ]:
# consultar os anos das vendas
df['data_venda'].dt.year.unique()

array([2024], dtype=int32)

Como as vendas ocorreram todas no mesmo ano não é necessário continuar usando o ano nas análises.

In [9]:
# criar a coluna de faturamento de cada venda
df['faturamento'] = df['quantidade'] * df['preco_unitario']

df.head()

,id_venda,data_venda,cliente,produto,categoria,quantidade,preco_unitario,vendedor,cidade,estado,faturamento
0,1,2024-09-27,Pietra,Mouse,Eletrônicos,3,84.74,Ana,Belo Horizonte,RJ,254.22
1,2,2024-01-21,Vinicius,Monitor,Eletrônicos,3,950.34,Bruno,São Paulo,RS,2851.02
2,3,2024-08-02,Yago,Mouse,Eletrônicos,3,79.13,Eduardo,Curitiba,SP,237.39
3,4,2024-04-09,Lara,Notebook,Eletrônicos,3,3471.47,Ana,Rio de Janeiro,RJ,10414.41
4,5,2024-05-29,Levi,Fone,Eletrônicos,4,208.32,Eduardo,São Paulo,RS,833.28


In [15]:
# criar uma função para tratar a exibição de valores de moeda
# essa função deve alterar apenas a exibição, sem modificar o valor no dataframe (que permanece como float)

def exibe_moeda(valor, pos=None):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


# testando a função
df.head().style.format({'faturamento': exibe_moeda})

,id_venda,data_venda,cliente,produto,categoria,quantidade,preco_unitario,vendedor,cidade,estado,faturamento
0,1,2024-09-27 00:00:00,Pietra,Mouse,Eletrônicos,3,84.740000,Ana,Belo Horizonte,RJ,"R$ 254,22"
1,2,2024-01-21 00:00:00,Vinicius,Monitor,Eletrônicos,3,950.340000,Bruno,São Paulo,RS,"R$ 2.851,02"
2,3,2024-08-02 00:00:00,Yago,Mouse,Eletrônicos,3,79.130000,Eduardo,Curitiba,SP,"R$ 237,39"
3,4,2024-04-09 00:00:00,Lara,Notebook,Eletrônicos,3,3471.470000,Ana,Rio de Janeiro,RJ,"R$ 10.414,41"
4,5,2024-05-29 00:00:00,Levi,Fone,Eletrônicos,4,208.320000,Eduardo,São Paulo,RS,"R$ 833,28"


In [16]:
# criar uma coluna para armazenar exclusivamente o mês da compra

# primeiro um dicionário para traduzir os nomes dos meses para português
meses_pt={
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}

# extrair o mês da data e já converter usando o dicionário
df['mes'] = df['data_venda'].dt.month.map(meses_pt)

# conferir o resultado
df.head()

,id_venda,data_venda,cliente,produto,categoria,quantidade,preco_unitario,vendedor,cidade,estado,faturamento,mes
0,1,2024-09-27,Pietra,Mouse,Eletrônicos,3,84.74,Ana,Belo Horizonte,RJ,254.22,Setembro
1,2,2024-01-21,Vinicius,Monitor,Eletrônicos,3,950.34,Bruno,São Paulo,RS,2851.02,Janeiro
2,3,2024-08-02,Yago,Mouse,Eletrônicos,3,79.13,Eduardo,Curitiba,SP,237.39,Agosto
3,4,2024-04-09,Lara,Notebook,Eletrônicos,3,3471.47,Ana,Rio de Janeiro,RJ,10414.41,Abril
4,5,2024-05-29,Levi,Fone,Eletrônicos,4,208.32,Eduardo,São Paulo,RS,833.28,Maio


In [17]:
# criar uma coluna para armazenar exclusivamente o dia da semana da compra

# novamente, primeiro um dicionário para converter o dia da semana para português
dias_semana_pt={
    0:'Segunda-feira', 1:'Terça-feira',
    2:'Quarta-feira', 3:'Quinta-feira', 4:'Sexta-feira',
    5:'Sábado', 6:'Domingo'
}

# extrair o dia da semana de cada data, já traduzindo
df['dia_semana'] = df['data_venda'].dt.day_of_week.map(dias_semana_pt)

# conferir o resultado
df.head()

,id_venda,data_venda,cliente,produto,categoria,quantidade,preco_unitario,vendedor,cidade,estado,faturamento,mes,dia_semana
0,1,2024-09-27,Pietra,Mouse,Eletrônicos,3,84.74,Ana,Belo Horizonte,RJ,254.22,Setembro,Sexta-feira
1,2,2024-01-21,Vinicius,Monitor,Eletrônicos,3,950.34,Bruno,São Paulo,RS,2851.02,Janeiro,Domingo
2,3,2024-08-02,Yago,Mouse,Eletrônicos,3,79.13,Eduardo,Curitiba,SP,237.39,Agosto,Sexta-feira
3,4,2024-04-09,Lara,Notebook,Eletrônicos,3,3471.47,Ana,Rio de Janeiro,RJ,10414.41,Abril,Terça-feira
4,5,2024-05-29,Levi,Fone,Eletrônicos,4,208.32,Eduardo,São Paulo,RS,833.28,Maio,Quarta-feira


Aqui se conclui a preparação dos dados. Assim, pode-se iniciar as análises.

## Evolução mensal de vendas

Inicialmente, uma visão gráfica sobre como o total de vendas se comportou ao longo do ano.

In [25]:
# gráfico de evolução do total de vendas ao longo do ano

# um novo dataset, exclusivo para ser usado nesse gráfico
df_evolucao_mensal = df.groupby('mes')['faturamento'].sum()

# arrumar a coluna de meses para que seja em formato string
df_evolucao_mensal['mes'] = df_evolucao_mensal['mes'].astype(str)

# conferir o resultado
df_evolucao_mensal


KeyError: 'mes'

In [ ]:

# instanciar o gráfico
fig, ax = plt.subplots()

# tipo de gráfico e eixos de análise
ax.plot(
    df['mes'],
    df['faturamento'].sum()
)


plt.show()